# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [24]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [25]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [26]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [27]:
load_dotenv()

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None


### VectorDB Instance

In [28]:
chroma_client = chromadb.PersistentClient(path="chromadb")


### Collection

In [29]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)


In [30]:
# Safely get or create the collection (prevents "already exists" error)
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

print("Collection ready:", collection.name)


Collection ready: udaplay


### Add documents

In [31]:
# Make sure you have a directory "games"
data_dir = "games"

documents = []
metadatas = []
ids = []

# Load and prepare all documents first
for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    documents.append(content)
    metadatas.append(game)
    ids.append(doc_id)

# 🔥 Reset collection safely (prevents duplicate ID errors)
chroma_client.delete_collection("udaplay")

collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

# Add all documents at once
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

# Run semantic search AFTER adding all documents
results = collection.query(
    query_texts=["Best racing game for PlayStation"],
    n_results=3
)

print(results)


{'ids': [['001', '003', '002']], 'embeddings': None, 'documents': [['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', '[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.', "[PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson."]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'YearOfRelease': 1997, 'Name': 'Gran Turismo', 'Platform': 'PlayStation 1', 'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.', 'Publisher': 'Sony Computer Entertainment', 'Genre': 'Racing'}, {'Publisher': 'Sony Computer Entertainment', 'Platform': 'PlayStation 3', 'YearOfRe